In [1]:
import nibabel as nib
import numpy as np
import pandas as pd


In [2]:

# Load the GIFTI label file
fpath = "brain_rendering_code/my_schaefer_tian_files/Schaefer200_Tian_S2.L.32k_fs_LR.label.gii"
img = nib.load(fpath)

# --- Metadata ---
print("=== File Metadata ===")
for key, val in img.meta.metadata.items():
    print(f"  {key}: {val[:80] if len(val) > 80 else val}")


=== File Metadata ===
  AnatomicalStructurePrimary: CortexLeft
  ParentProvenance: Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii:
/Applicati
  ProgramProvenance: Connectome Workbench
Type: Command Line Application
Version: 2.1.0
Qt Compiled V
  Provenance: /Applications/wb_view.app/Contents/usr/bin/../exe/exe_wb_command -cifti-separate
  WorkingDirectory: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcella


/var/folders/n7/g8bq81dn5879t8lq7djjjst40000gn/T/ipykernel_39782/1197461750.py:7: DeprecationWarning: metadata property deprecated. Use GiftiMetaData object as dict or pass to dict() for a standard dictionary.

* deprecated from version: 4.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 6.0
  for key, val in img.meta.metadata.items():


In [7]:
# --- Label table ---
print("\n=== Label Table ===")
label_table = img.labeltable.labels  # list of GiftiLabel objects
rows = []
for label in label_table:
    rows.append({
        "key": label.key,
        "name": label.label,
        "rgba": (round(label.red,3), round(label.green,3), round(label.blue,3), round(label.alpha,3))
    })
df_labels = pd.DataFrame(rows)
print(f"Total labels: {len(df_labels)}")
print(df_labels.to_string())



=== Label Table ===
Total labels: 233
     key                                   name                        rgba
0      0                                    ???        (1.0, 1.0, 1.0, 0.0)
1      1                                aHIP-rh  (0.565, 0.965, 0.176, 1.0)
2      2                                pHIP-rh  (0.133, 0.647, 0.231, 1.0)
3      3                                lAMY-rh  (0.737, 0.376, 0.298, 1.0)
4      4                                mAMY-rh  (0.957, 0.757, 0.796, 1.0)
5      5                              THA-DP-rh   (0.69, 0.827, 0.596, 1.0)
6      6                              THA-VP-rh   (0.761, 0.376, 0.02, 1.0)
7      7                              THA-VA-rh  (0.039, 0.812, 0.639, 1.0)
8      8                              THA-DA-rh  (0.306, 0.365, 0.012, 1.0)
9      9                           NAc-shell-rh   (0.439, 0.58, 0.902, 1.0)
10    10                            NAc-core-rh  (0.565, 0.208, 0.906, 1.0)
11    11                                 pGP-rh  

In [8]:

# --- Data array ---
print("\n=== Data Array ===")
data = img.darrays[0].data
print(f"Shape: {data.shape}  (vertices on 32k LH surface)")
print(f"dtype: {data.dtype}")
print(f"Unique parcel keys assigned: {np.unique(data)}")
print(f"Number of non-zero (labelled) vertices: {np.sum(data > 0)}")
print(f"Number of unlabelled (key=0) vertices: {np.sum(data == 0)}")

# --- Vertices per parcel ---
print("\n=== Vertex counts per parcel ===")
keys, counts = np.unique(data, return_counts=True)
df_counts = pd.DataFrame({"key": keys, "vertex_count": counts})
df_counts = df_counts.merge(df_labels[["key","name"]], on="key", how="left")
print(df_counts.to_string(index=False))

# --- Quick summary by region type ---
subcortical = df_labels[df_labels["key"].between(1, 32)]
cortical_lh = df_labels[df_labels["name"].str.startswith("7Networks_LH")]
cortical_rh = df_labels[df_labels["name"].str.startswith("7Networks_RH")]
print(f"\nSubcortical labels (Tian S2):  {len(subcortical)}")
print(f"Cortical LH parcels (Schaefer): {len(cortical_lh)}")
print(f"Cortical RH parcels (Schaefer): {len(cortical_rh)}")



=== Data Array ===
Shape: (32492,)  (vertices on 32k LH surface)
dtype: int32
Unique parcel keys assigned: [  0  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49
  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67
  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85
  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103
 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121
 122 123 124 125 126 127 128 129 130 131 132]
Number of non-zero (labelled) vertices: 29301
Number of unlabelled (key=0) vertices: 3191

=== Vertex counts per parcel ===
 key  vertex_count                                 name
   0          3191                                  ???
  33           256                   7Networks_LH_Vis_1
  34           420                   7Networks_LH_Vis_2
  35           270                   7Networks_LH_Vis_3
  36           226                   7Networks_LH_Vis_4
  37           276   